# 3.5 Email Agent — Capstone: Everything Together

## What this notebook demonstrates

This is the **Module 3 capstone**. It combines every advanced concept from this module into one realistic application:

| Concept | Where it's used |
|---|---|
| **Custom state** | `AuthenticatedState` — tracks login status across turns |
| **Runtime context** | `EmailContext` — injects credentials per request |
| **State-writing tool** | `authenticate` — writes `authenticated: True/False` to state |
| **Dynamic tools** | `@wrap_model_call` — exposes different tools pre/post login |
| **Dynamic prompt** | `@dynamic_prompt` — changes persona pre/post login |
| **Human-in-the-Loop** | `HumanInTheLoopMiddleware` — pauses before sending emails |

### The application flow

```
User message arrives
        │
        ▼
  authenticated? ──No──→  Only tool: authenticate
        │                 System prompt: "You help users authenticate"
        │                 User must provide email + password
        │
       Yes
        │
        ▼
  Tools: check_inbox, send_email
  System prompt: "You can check inbox and send emails"
        │
        ▼
  send_email called? ──→  HITL interrupt — human approves/rejects
        │
        ▼
   Final response
```

### Why this architecture matters

This pattern — authenticate → unlock capabilities → HITL for sensitive actions — is the blueprint for secure agentic applications:
- Banking agents (verify identity → allow transfers → require approval)
- HR agents (verify employee → allow sensitive data → require manager sign-off)
- Admin tools (verify admin role → allow dangerous operations → require confirmation)

In [1]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================

from dotenv import load_dotenv

load_dotenv()

True

In [4]:
# ============================================================
# CELL 2: Define the Context Schema (User Credentials)
# ============================================================
# EmailContext holds the credentials injected at request time.
#
# In production:
#   - These come from your auth system, NOT from the user's message
#   - The user types their password in a login form
#   - Your backend verifies it and stores the session
#   - On each agent request, you inject EmailContext from the session
#   - The credentials NEVER appear in the conversation history
#
# The authenticate tool compares what the user CLAIMS (in their message)
# against what you INJECTED (in context). If they match, authentication succeeds.
# This simulates a real auth flow without a real auth server.

from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"  # The real email (server-side)
    password: str = "password123"              # The real password (server-side)

In [5]:
# ============================================================
# CELL 3: Define the Custom State Schema
# ============================================================
# AuthenticatedState adds one boolean field to the default AgentState:
#   authenticated: bool
#
# This field:
#   - Starts as unset (no value) for a new thread
#   - Is set to True by the authenticate tool on successful login
#   - Is set to False by the authenticate tool on failed login
#   - Persists across turns via InMemorySaver — once logged in, stays logged in
#   - Is read by @wrap_model_call and @dynamic_prompt to control behaviour
#
# The authenticated field is the single source of truth for whether
# this session has been authenticated. All security decisions flow from it.

from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool   # Set by authenticate tool; controls dynamic tools/prompt

In [32]:
# ============================================================
# CELL 4: Define the Three Tools
# ============================================================
# Tool 1: check_inbox (post-auth only)
#   → Simulates reading the inbox
#   → In production: calls Gmail/Outlook API
#   → Returns a fake email from Jane for demo purposes
#
# Tool 2: send_email (post-auth only, HITL-protected)
#   → Simulates sending an email
#   → In production: calls email sending API
#   → HITL middleware will PAUSE before this executes
#
# Tool 3: authenticate (pre-auth only)
#   → Compares user-provided credentials against context
#   → Returns Command to write to state:
#       True  → login succeeded → dynamic_tool_call will unlock inbox/send
#       False → login failed   → agent tells user to try again
#   → Uses runtime.context for the REAL credentials (server-side)
#   → Uses email/password arguments for what the USER CLAIMED (agent-extracted)
#   → runtime.tool_call_id is needed to link the ToolMessage back to the call

from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    # In production: fetch from Gmail/Outlook API
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    # In production: call Gmail/SendGrid API
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    # Compare user's claim (from message) against real credentials (from context)
    print("inside authenticate", email, password)
    if runtime.tool_call_id:
        return Command(update={})
    if email == runtime.context.email_address and password == runtime.context.password:
        # Match → write authenticated=True to state
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        # Mismatch → write authenticated=False to state
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

In [33]:
# ============================================================
# CELL 5: Define Dynamic Tool Selection (Auth-Gated)
# ============================================================
# This middleware reads the authenticated field from state and
# presents a completely different tool set based on its value:
#
# Not authenticated:
#   tools = [authenticate]  → only login is possible
#   The model cannot call check_inbox or send_email even if asked
#
# Authenticated:
#   tools = [check_inbox, send_email]  → email capabilities unlocked
#   authenticate is no longer available (user is already logged in)
#
# request.state.get("authenticated") uses .get() defensively:
#   → Returns None if the field hasn't been set yet (first turn)
#   → None is falsy → agent gets [authenticate] → correct for first turn
#
# This override runs before EVERY LLM call, so if the user
# authenticates mid-conversation, the next LLM call immediately
# gets the expanded tool set.

from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(
    request: ModelRequest, 
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Allow inbox/send tools only after successful authentication"""

    authenticated = request.state.get("authenticated")   # None if not yet set
    
    if authenticated:
        tools = [check_inbox, send_email]   # Post-login: email capabilities
    else:
        tools = [authenticate] # Pre-login: only login allowed
    print("dynamic_tool_call",tools)
    request = request.override(tools=tools) 
    return handler(request)

In [39]:
# ============================================================
# CELL 6: Define Dynamic Prompt (Auth-Aware Persona)
# ============================================================
# The system prompt changes based on authentication status:
#
# Not authenticated:
#   → Persona: "authenticate users"
#   → Agent focuses on getting credentials, not on email tasks
#   → This prevents the agent from rambling about emails before login
#
# Authenticated:
#   → Persona: "check inbox and send emails"
#   → Agent focuses on email tasks
#   → Authentication tool is no longer mentioned or offered
#
# Note: the @dynamic_prompt function is named 'dynamic_prompt' locally,
# which shadows the imported decorator. This is intentional — the
# decorated function is the one passed to middleware=[].

from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = "You are a helpful assistant that can check the inbox and send emails."
unauthenticated_prompt = "You are a helpful assistant that can authenticate users by calling authenticate tool."

@dynamic_prompt
def dynamic_prompt_fn(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt  # Email persona after login
    else:
        print("unauthenticated_prompt",unauthenticated_prompt)
        return unauthenticated_prompt  # Auth persona before login

In [43]:
# ============================================================
# CELL 7: Create the Full Email Agent
# ============================================================
# All three middleware run in order for every LLM call:
#   1. dynamic_tool_call  → sets the tool list (auth-gated)
#   2. dynamic_prompt_fn  → sets the system prompt (auth-aware)
#   3. HumanInTheLoopMiddleware → pauses before send_email
#
# The middleware order matters:
#   dynamic_tool_call must run before HumanInTheLoopMiddleware
#   because HITL uses the tool list to know which tools to watch.
#   If dynamic_tool_call runs after HITL, HITL might try to interrupt
#   on authenticate (which it shouldn't).
#
# HumanInTheLoopMiddleware interrupt_on:
#   authenticate: False  → never pause for authentication calls
#   check_inbox:  False  → never pause for inbox reads (safe)
#   send_email:   True   → ALWAYS pause before sending
#
# tools=[authenticate, check_inbox, send_email] is the MAXIMUM set.
# dynamic_tool_call reduces this per-request based on auth status.

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    "gpt-5",
    tools=[authenticate, check_inbox, send_email],  # Maximum tool set
    checkpointer=InMemorySaver(),   # Required for state persistence AND HITL resume
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call,          # 1st: filter tools by auth status
        dynamic_prompt_fn,          # 2nd: set prompt by auth status
        HumanInTheLoopMiddleware(   # 3rd: pause before send_email
            interrupt_on={
                "authenticate": False,  # Auto-execute login
                "check_inbox":  False,  # Auto-execute read
                "send_email":   True,   # PAUSE — needs human approval
            })
    ]
)

In [45]:
# ============================================================
# CELL 8: Turn 1 — "draft 1" Triggers the Full Flow
# ============================================================
# "draft 1" is intentionally cryptic — it simulates a user who
# has a pre-configured workflow. The agent (unauthenticated persona)
# will:
#   1. See the user hasn't authenticated
#   2. Read credentials from EmailContext
#   3. Call authenticate(email="julie@...", password="password123")
#   4. Receive Command(authenticated=True)
#   5. Now authenticated → switches to email persona
#   6. Calls check_inbox() → reads the Jane email
#   7. Drafts a reply to Jane
#   8. Attempts send_email() → INTERRUPTED by HITL
#   9. Returns with '__interrupt__' key
#
# context=EmailContext() injects the real credentials server-side.
# The agent extracts them from context (not from the user's message)
# and uses them in the authenticate call.

from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="draft 1")]},
    context=EmailContext(),   # Real credentials injected server-side
    config=config
)

print(response['messages'][-1].content)

dynamic_tool_call [StructuredTool(name='authenticate', description='Authenticate the user with the given email and password', args_schema=<class 'langchain_core.utils.pydantic.authenticate'>, func=<function authenticate at 0x10ae22700>)]
unauthenticated_prompt You are a helpful assistant that can authenticate users by calling authenticate tool.
inside authenticate prudhvi.akella@yahoo.com 12345


/Users/akellaprudhvi/mystuff/Course/GenAI-Course-Modules/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=EmailContext(email_addres... password='password123'), input_type=EmailContext])
  return self.__pydantic_serializer__.to_python(


ValueError: Expected to have a matching ToolMessage in Command.update for tool 'authenticate', got: []. Every tool call (LLM requesting to call a tool) in the message history MUST have a corresponding ToolMessage. You can fix it by modifying the tool to return `Command(update=[ToolMessage("Success", tool_call_id=tool_call_id), ...], ...)`.

In [37]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='draft 1', additional_kwargs={}, response_metadata={}, id='02ca43dd-ec84-42b1-8caa-441aea865cfd'),
              AIMessage(content='I’m not sure what you mean by “draft 1.” Could you tell me what you’d like to draft (email, report, proposal, note, etc.) and the details?\n\nHelpful details to share:\n- Purpose and audience\n- Key points or sections to include\n- Desired length or word count\n- Tone (formal, casual, persuasive, etc.)\n\nIf you actually want to sign in, please provide your email and password and I can help with authentication.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 616, 'prompt_tokens': 147, 'total_tokens': 763, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 

In [14]:
# ============================================================
# CELL 9: Inspect the Proposed Email
# ============================================================
# The agent is paused at the send_email tool call.
# Inspect the email body it drafted — is it appropriate?
# Does it have the right tone? Is the recipient correct?

print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

KeyError: '__interrupt__'

In [ ]:
# ============================================================
# CELL 10: Approve and Resume
# ============================================================
# After reviewing the draft in Cell 9, we approve it.
# Command(resume={"decisions": [{"type": "approve"}]}) resumes the graph.
# The same thread_id="1" config is used — LangGraph loads the
# paused state from InMemorySaver and continues from the interrupt point.
#
# The graph will:
#   1. Execute send_email() with the approved body
#   2. Agent generates the final confirmation message

from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # Or "reject" to re-draft
    ), 
    config=config   # Same thread_id — resumes the paused graph
)

print(response["messages"][-1].content)

In [ ]:
# ============================================================
# CELL 11: Inspect the Full Final State
# ============================================================
# The full response dict shows:
#   response['authenticated']  → True (persisted from the authenticate call)
#   response['messages']       → full conversation including auth, inbox read, send
#
# The authentication, inbox read, email draft, approval, and send
# are all captured in the message history — a complete audit trail.

from pprint import pprint

pprint(response)

---
## Summary — What Makes This Production-Grade

| Feature | Implementation | Why it matters |
|---|---|---|
| **Authentication** | `authenticate` tool writes to state | Session persists across turns |
| **Credential security** | Injected via context, never in messages | Credentials don't appear in conversation history |
| **Tool gating** | `dynamic_tool_call` reads `authenticated` state | Unauthenticated users can't access email tools |
| **Adaptive persona** | `dynamic_prompt_fn` changes based on auth | Agent behaviour matches its actual capabilities |
| **Send protection** | HITL interrupts before `send_email` | Human reviews every outgoing email |
| **Full audit trail** | InMemorySaver persists all messages | Complete history of auth, reads, sends |

### The three-layer security model

```
Layer 1: Authentication
  → Credentials verified before any email capability is unlocked

Layer 2: Dynamic tool restriction
  → Even authenticated users can't access tools outside their role

Layer 3: Human-in-the-Loop
  → High-stakes actions (send email) require explicit human approval
```

This same three-layer pattern applies to any high-stakes agentic system.